In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Входи та виходи Estimator

<Accordion>
<AccordionItem title="Версії пакетів">

Код на цій сторінці було розроблено з використанням таких вимог.
Рекомендуємо використовувати ці або новіші версії.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Ця сторінка дає огляд входів та виходів примітива Qiskit Runtime Estimator, який виконує навантаження на обчислювальних ресурсах IBM Quantum&reg;. Estimator дозволяє ефективно визначати векторизовані навантаження, використовуючи структуру даних, що називається [**Primitive Unified Bloc (PUB)**](/guides/primitive-input-output#pubs). Вони використовуються як вхідні дані для методу [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) примітива Estimator, який виконує визначене навантаження як завдання. Потім, після завершення завдання, результати повертаються у форматі, що залежить від обох PUB, використовуваних як вхідні дані, а також від опцій середовища виконання, вказаних примітивом.
## Входи
Кожен PUB має такий формат:

(`<одна схема>`, `<один або кілька спостережуваних>`, `<необов'язкові одне або кілька значень параметрів>`, `<необов'язкова точність>`),

Необов'язкові `parameter values` можуть бути списком або одним параметром. Елементи зі спостережуваних та значень параметрів комбінуються відповідно до правил трансляції NumPy, як описано в темі [Входи та виходи примітивів](primitive-input-output#broadcasting-rules), і одна оцінка очікуваного значення повертається для кожного елемента трансльованої форми.

> **Note:** Якщо вхідні дані містять вимірювання, вони ігноруються.

Для примітива Estimator PUB може містити не більше чотирьох значень:
- Одна `QuantumCircuit`, яка може містити один або кілька об'єктів [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)
- Список одного або кількох спостережуваних, що вказують очікувані значення для оцінки, впорядковані в масив (наприклад, одне спостережуване, представлене як 0-d масив, список спостережуваних як 1-d масив тощо). Дані можуть бути в будь-якому з форматів `ObservablesArrayLike`, таких як `Pauli`, `SparsePauliOp`, `PauliList` або `str`.
   > **Note:** - Комутуючі спостережувані **в одному і тому ж PUB** групуються разом за допомогою [цього методу](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting).
>    - Комутуючі спостережувані в різних PUB, навіть якщо вони мають однакову схему, не оцінюються за допомогою одного вимірювання. Кожен PUB представляє різний базис для вимірювання, і тому для кожного PUB потрібні окремі вимірювання.
>    - Щоб гарантувати, що комутуючі спостережувані оцінюються за допомогою одного вимірювання, групуй їх у тому ж PUB.
- Колекція значень параметрів для прив'язки схеми. Це може бути вказано як єдиний об'єкт, схожий на масив, де останній індекс — над об'єктами `Parameter` схеми, або пропущено (або еквівалентно, встановлено на `None`), якщо схема не має об'єктів `Parameter`.
- (Необов'язково) Цільова точність для оцінки очікуваних значень
---

Наступний код демонструє приклад набору векторизованих входів до примітива `Estimator` та виконує їх на backend IBM&reg; як єдиний об'єкт `RuntimeJobV2`.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

## Виходи
Після надсилання одного або кількох PUB до QPU для виконання та успішного завершення завдання дані повертаються як об'єкт-контейнер [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), доступний шляхом виклику методу `RuntimeJobV2.result()`.

`PrimitiveResult` містить ітерований список об'єктів [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult), що містять результати виконання для кожного PUB.

Кожен елемент цього списку відповідає кожному PUB, надісланому до методу `run()` примітива (наприклад, завдання, надіслане з 20 PUB, поверне об'єкт `PrimitiveResult`, що містить список з 20 об'єктів `PubResult`, кожен відповідний своєму PUB).

Кожен `PubResult` для примітива Estimator містить щонайменше масив очікуваних значень (`PubResult.data.evs`) та пов'язані стандартні відхилення (або `PubResult.data.stds`, або `PubResult.data.ensemble_standard_error` залежно від використовуваного `resilience_level`), але може містити більше даних залежно від вказаних опцій пом'якшення помилок.

Кожен об'єкт `PubResult` має як атрибут `data`, так і атрибут `metadata`.
    - Атрибут `data` — це налаштований [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin), що містить фактичні значення вимірювань, стандартні відхилення тощо.
    - `DataBin` має різні атрибути залежно від форми або структури пов'язаного PUB, а також від опцій пом'якшення помилок, вказаних примітивом, використовуваним для надсилання завдання (наприклад, [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) або [PEC](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)).
    - Атрибут `metadata` містить інформацію про опції середовища виконання та пом'якшення помилок, використовувані (пояснено пізніше в розділі [Метадані результату](#result-metadata) цієї сторінки).

Нижче наведено візуальний контур структури даних `PrimitiveResult` для виходу Estimator:

    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```

Простіше кажучи, одне завдання повертає об'єкт `PrimitiveResult` та містить список з одного або кількох об'єктів `PubResult`. Ці об'єкти `PubResult` потім зберігають дані вимірювань для кожного PUB, надісланого до завдання.

Наступний фрагмент коду описує формат `PrimitiveResult` (та пов'язаного `PubResult`) для завдання, створеного вище.

In [ ]:
print(
    f"The result of the submitted job had {len(result)} "
    f"PUBs and has a value:\n {result}\n"
)
print(
    "The associated PubResult of this job has the following data bins:\n "
    "{result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets"
    "having shape (100, 2), where 2 is the number of parameters in the "
    "circuit, combined with our array of observables having shape (3, 1). \n"
)
with np.printoptions(threshold=200):
    print(
        "The expectation values measured from this PUB are: \n"
        "{result[0].data.evs}\n"
    )

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

#### How the Estimator primitive calculates error

In addition to the estimate of the mean of the observables passed in the input PUBs (the `evs` field of the `DataBin`), Estimator also attempts to deliver an estimate of the error associated with those expectation values. All Estimator queries will populate the `stds` field with a quantity like the standard error of the mean for each expectation value, but some error mitigation options produce additional information, such as `ensemble_standard_error`.

Consider a single observable $\mathcal{O}$. In the absence of [ZNE](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), you can think of each shot of the Estimator execution as providing a point estimate of the expectation value $\langle \mathcal{O} \rangle$. If the pointwise estimates are in a vector `Os`, then the value returned in `ensemble_standard_error` is equivalent to the following (in which $\sigma_{\mathcal{O}}$ is the [standard deviation of the expectation value](/docs/api/qiskit/qiskit.primitives.BackendEstimatorV2) estimate and $N_{shots}$ is the number of shots):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

which treats all shots as part of a single ensemble. If you requested gate [twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) (`twirling.enable_gates = True`), you can sort the pointwise estimates of $\langle \mathcal{O} \rangle$ into sets that share a common twirl. Call these sets of estimates `O_twirls`, and there are `num_randomizations` (number of twirls) of them. Then `stds` is the standard error of the mean of `O_twirls`, as in

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

where $\sigma_{\mathcal{O}}$ is the standard deviation of `O_twirls` and $N_{twirls}$ is the number of twirls. When you do not enable twirling, `stds` and `ensemble_standard_error` are equal.

If you enable ZNE, then the `stds` described above become weights in a non-linear regression to an extrapolator model. What finally gets returned in the `stds` in this case is the uncertainty of the fit model evaluated at a noise factor of zero. When there is a poor fit, or large uncertainty in the fit, the reported `stds` can become very large. When ZNE is enabled, `pub_result.data.evs_noise_factors` and `pub_result.data.stds_noise_factors` are also populated, so that you can do your own extrapolation.

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [3]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'dynamical_decoupling' : {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'},
'twirling' : {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'},
'resilience' : {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False},
'version' : 2,

The metadata of the PubResult result is:
'shots' : 4096,
'target_precision' : 0.015625,
'circuit_metadata' : {},
'resilience' : {},
'num_randomizations' : 32,
